# SIGNAL-SHIFT
## Model Reliability Forensics: Detecting Hidden Data Drift Before Predictions Become Untrustworthy

**Project category:** Advanced Machine Learning · Data Science · Model Monitoring · Explainable AI

### The unusual question
Most ML projects ask: *How accurately can a model predict?*

SIGNAL-SHIFT asks a harder operational question:

> **When the world changes, how early can we detect that a model's predictions may no longer be trustworthy?**

This notebook builds a complete research prototype that:
- creates a realistic, explicitly synthetic network-traffic dataset;
- trains a baseline intrusion classifier;
- simulates changing traffic behaviour across time windows;
- detects feature drift using Population Stability Index (PSI) and Kolmogorov–Smirnov tests;
- measures performance degradation when labels become available;
- monitors prediction confidence and alert rates;
- ranks features contributing to drift;
- combines signals into a transparent model-health score;
- produces a final incident-style reliability report.

**Important:** All generated observations are synthetic. This is an educational monitoring prototype, not a production security detector. Drift is not automatically evidence of model failure; it is a signal to investigate.

## 1. Research questions and hypotheses

**RQ1.** Can statistical monitoring detect changes in incoming network traffic before model performance is evaluated?

**RQ2.** Do feature-level drift signals identify which parts of the input distribution changed?

**RQ3.** Does a rise in model uncertainty coincide with a decline in predictive performance?

**RQ4.** Can a transparent health score summarize multiple monitoring signals without hiding their individual meanings?

**Hypotheses**
- H1: Injected distribution shifts will increase PSI and/or KS drift signals.
- H2: Severe shifts in predictive features will reduce held-out performance.
- H3: confidence and error-rate monitoring provide complementary evidence to feature drift.
- H4: A combined score is more informative for triage than any single metric.

These hypotheses are tested on controlled synthetic data; they are not universal claims.

In [ ]:
# 2. Environment and reproducibility
import sys, warnings, math, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from scipy.stats import ks_2samp

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 3. Generate a synthetic baseline network-traffic population

The features are inspired by common traffic-analysis variables: packet size, inter-arrival time, connection duration, failed-connection rate, destination service count, root activity, and byte ratio.

The target is generated from a transparent nonlinear rule plus noise. This lets us test monitoring logic without pretending to have real network telemetry.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def make_traffic(n, seed=42, regime="baseline"):
    r = np.random.default_rng(seed)

    # Baseline feature distributions
    packet_size = r.lognormal(mean=4.0, sigma=0.42, size=n).clip(5, 500)
    inter_arrival = r.gamma(shape=2.0, scale=0.22, size=n).clip(0.001, 8)
    duration = r.lognormal(mean=0.5, sigma=0.8, size=n).clip(0.01, 60)
    srv_error_rate = r.beta(0.7, 10, size=n)
    dst_host_srv_count = r.poisson(42, size=n).clip(0, 180)
    num_root = r.choice([0, 1, 2, 3, 4], size=n, p=[.43, .39, .14, .035, .005])
    byte_ratio = r.lognormal(mean=0.0, sigma=0.55, size=n).clip(.05, 12)
    packet_count = r.poisson(18, size=n).clip(1, 160)

    if regime == "volume_shift":
        packet_size = (packet_size * 1.45).clip(5, 900)
        packet_count = r.poisson(32, size=n).clip(1, 240)
    elif regime == "timing_shift":
        inter_arrival = (inter_arrival * 2.2 + .15).clip(.001, 15)
        duration = (duration * 1.8).clip(.01, 120)
    elif regime == "error_shift":
        srv_error_rate = np.clip(r.beta(2.2, 3.0, size=n), 0, 1)
        dst_host_srv_count = r.poisson(22, size=n).clip(0, 180)
    elif regime == "mixed_shift":
        packet_size = (packet_size * 1.3).clip(5, 900)
        inter_arrival = (inter_arrival * 1.7 + .1).clip(.001, 15)
        srv_error_rate = np.clip(r.beta(1.8, 4.0, size=n), 0, 1)
        dst_host_srv_count = r.poisson(25, size=n).clip(0, 180)
        num_root = r.choice([0,1,2,3,4], size=n, p=[.25,.35,.25,.12,.03])

    # Synthetic risk mechanism; labels are not real-world attack labels.
    z = (
        -3.0
        + 4.5 * srv_error_rate
        + 0.018 * packet_size
        + 0.75 * (num_root >= 2)
        + 0.012 * (180 - dst_host_srv_count)
        + 0.22 * np.log1p(packet_count)
        - 0.35 * np.log1p(inter_arrival)
        + 0.18 * np.abs(np.log(byte_ratio))
    )
    probability = sigmoid(z)
    y = r.binomial(1, probability)

    X = pd.DataFrame({
        "packet_size": packet_size,
        "inter_arrival_time": inter_arrival,
        "connection_duration": duration,
        "srv_error_rate": srv_error_rate,
        "dst_host_srv_count": dst_host_srv_count,
        "num_root": num_root,
        "byte_ratio": byte_ratio,
        "packet_count": packet_count
    })
    return X, pd.Series(y, name="synthetic_attack_label"), probability

X_base, y_base, latent_risk = make_traffic(6000, seed=RANDOM_STATE)
baseline = X_base.copy()
baseline["label"] = y_base

display(baseline.head())
print("Rows:", len(baseline), "| Features:", X_base.shape[1])
print("Synthetic positive class rate:", round(y_base.mean(), 3))

## 4. Explore the baseline population

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, X_base.columns):
    sns.histplot(X_base[col], kde=True, ax=ax, color="#3977b8")
    ax.set_title(col.replace("_", " ").title())
plt.suptitle("Baseline feature distributions", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 6))
sns.heatmap(baseline.corr(numeric_only=True), cmap="vlag", center=0, annot=True, fmt=".2f")
plt.title("Baseline feature and target correlation")
plt.tight_layout()
plt.show()

## 5. Train a baseline model and freeze the reference population

We use a train/test split. The baseline test set represents the initial reference distribution. Later monitoring windows are evaluated against the training reference and, separately, against known synthetic labels.

For a real deployment, the reference set must be chosen carefully, with temporal and entity-level leakage controls.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y_base, test_size=0.25, random_state=RANDOM_STATE, stratify=y_base
)

numeric_features = X_train.columns.tolist()
preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=350, min_samples_leaf=3,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])

model.fit(X_train, y_train)

def evaluate(y_true, prob, threshold=0.5):
    pred = (np.asarray(prob) >= threshold).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
    }

base_prob = model.predict_proba(X_test)[:, 1]
base_metrics = evaluate(y_test, base_prob)
display(pd.DataFrame([base_metrics], index=["Baseline holdout"]).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cm = confusion_matrix(y_test, base_prob >= 0.5)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Predicted 0", "Predicted 1"],
            yticklabels=["Actual 0", "Actual 1"])
axes[0].set_title("Baseline confusion matrix")
axes[1].hist(base_prob[y_test.values == 0], bins=25, alpha=.65, label="Class 0")
axes[1].hist(base_prob[y_test.values == 1], bins=25, alpha=.65, label="Class 1")
axes[1].set_title("Baseline predicted probability distributions")
axes[1].set_xlabel("Predicted probability")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Create time-ordered monitoring windows with controlled shifts

We simulate four incoming data windows:
- **Window 0:** stable baseline-like traffic.
- **Window 1:** packet-volume shift.
- **Window 2:** timing shift.
- **Window 3:** error-rate and service-count shift.
- **Window 4:** mixed shift.

Labels are included only because this is a controlled experiment. In real operations, labels may arrive late or never.

In [ ]:
regimes = ["baseline", "volume_shift", "timing_shift", "error_shift", "mixed_shift"]
windows = {}
window_labels = {}

for i, regime in enumerate(regimes):
    Xw, yw, _ = make_traffic(1800, seed=100+i, regime=regime)
    windows[f"Window {i}"] = Xw
    window_labels[f"Window {i}"] = yw

print("Monitoring windows:", list(windows))

In [ ]:
# Compare a selected feature across regimes
features_to_show = ["packet_size", "inter_arrival_time", "srv_error_rate", "dst_host_srv_count"]
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, feature in zip(axes.flat, features_to_show):
    for name, Xw in windows.items():
        sns.kdeplot(Xw[feature], ax=ax, label=name, fill=False)
    ax.set_title(feature.replace("_", " ").title())
    ax.legend(fontsize=8)
plt.suptitle("How the incoming population changes across time windows", fontsize=15)
plt.tight_layout()
plt.show()

## 7. Statistical drift detection: PSI and Kolmogorov–Smirnov

**Population Stability Index (PSI)** compares binned proportions between a reference and a current window.

\[
PSI = \sum_i (p_i-q_i)\ln\left(\frac{p_i}{q_i}\right)
\]

**KS test** compares two empirical distributions. Its p-value is sample-size-sensitive, so interpret it alongside effect size and domain relevance.

The thresholds below are conventional heuristics for demonstration, not universal operating standards.

In [ ]:
def psi(reference, current, bins=10, eps=1e-6):
    ref = np.asarray(reference, dtype=float)
    cur = np.asarray(current, dtype=float)
    # Fixed reference quantile bins, with infinite outer edges.
    edges = np.unique(np.quantile(ref[np.isfinite(ref)], np.linspace(0, 1, bins+1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts = np.histogram(ref, bins=edges)[0].astype(float)
    cur_counts = np.histogram(cur, bins=edges)[0].astype(float)
    p = np.clip(ref_counts / max(ref_counts.sum(), 1), eps, None)
    q = np.clip(cur_counts / max(cur_counts.sum(), 1), eps, None)
    return float(np.sum((q-p) * np.log(q/p)))

def drift_table(reference, current):
    rows=[]
    for col in reference.columns:
        a = reference[col].dropna()
        b = current[col].dropna()
        stat, pval = ks_2samp(a, b)
        val = psi(a, b)
        rows.append({"feature":col, "PSI":val, "KS_statistic":stat, "KS_pvalue":pval})
    out = pd.DataFrame(rows).sort_values("PSI", ascending=False)
    out["PSI_flag"] = pd.cut(out["PSI"], [-np.inf, .1, .25, np.inf],
                              labels=["Low", "Moderate", "High"])
    return out

reference = X_train.copy()
drift_results = {name: drift_table(reference, Xw) for name, Xw in windows.items()}

for name, table in drift_results.items():
    print("\\n", name)
    display(table.round(4))

In [ ]:
psi_matrix = pd.DataFrame({
    name: table.set_index("feature")["PSI"]
    for name, table in drift_results.items()
})
plt.figure(figsize=(11, 6))
sns.heatmap(psi_matrix, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title("Feature drift map (PSI): reference vs monitoring windows")
plt.xlabel("Monitoring window")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 8. Monitor model behaviour: confidence, alerts, and delayed labels

Feature drift alone does not prove that predictions have become worse. We therefore add model-behaviour monitoring.

For each window we calculate:
- mean predicted probability;
- fraction of predictions above the alert threshold;
- fraction of uncertain predictions near 0.5;
- performance metrics (available here because synthetic labels are known).

The uncertainty band is configurable and should be selected based on the intended operational risk.

In [ ]:
def monitor_window(name, Xw, yw, threshold=.5, uncertainty_band=.15):
    prob = model.predict_proba(Xw)[:, 1]
    metrics = evaluate(yw, prob, threshold)
    return {
        "window": name,
        "mean_probability": float(np.mean(prob)),
        "alert_rate": float(np.mean(prob >= threshold)),
        "uncertain_rate": float(np.mean(np.abs(prob-.5) <= uncertainty_band)),
        **metrics
    }

monitor_rows = [
    monitor_window(name, windows[name], window_labels[name])
    for name in windows
]
monitor_df = pd.DataFrame(monitor_rows)
display(monitor_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for metric, ax in zip(["f1", "roc_auc", "uncertain_rate"], axes):
    sns.lineplot(data=monitor_df, x="window", y=metric, marker="o", ax=ax)
    ax.set_title(metric.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=25)
axes[0].axhline(base_metrics["f1"], ls="--", color="black", alpha=.5, label="Baseline holdout")
axes[0].legend()
plt.suptitle("Model behaviour across changing data windows")
plt.tight_layout()
plt.show()

## 9. Drift attribution: which variables moved the most?

In [ ]:
top_drift = (pd.concat([t.assign(window=name) for name,t in drift_results.items()])
             .groupby("feature")["PSI"].max().sort_values(ascending=False))
plt.figure(figsize=(10, 5))
sns.barplot(x=top_drift.values, y=top_drift.index, color="#5b8fd1")
plt.title("Maximum observed PSI by feature across monitoring windows")
plt.xlabel("Maximum PSI")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(top_drift.rename("maximum_PSI").to_frame().round(4))

## 10. Transparent model-health score

A single score can help triage, but must not conceal the underlying signals. Here we define an **experimental health index** from three normalized components:

- average capped PSI across features;
- increase in uncertain predictions relative to baseline;
- relative F1 degradation relative to baseline holdout.

The score is a demonstrator, not a validated reliability probability. Always inspect the component metrics and drift table.

In [ ]:
baseline_uncertain = float(np.mean(np.abs(base_prob-.5) <= .15))
baseline_f1 = base_metrics["f1"]

health_rows=[]
for name, Xw in windows.items():
    max_psi = float(drift_results[name]["PSI"].mean())
    obs = monitor_df.loc[monitor_df.window == name].iloc[0]
    drift_component = min(max_psi / .5, 1.0)
    uncertainty_component = min(max(obs["uncertain_rate"]-baseline_uncertain,0) / .30, 1.0)
    performance_drop = max((baseline_f1-obs["f1"]) / max(baseline_f1,1e-6), 0)
    performance_component = min(performance_drop / .50, 1.0)
    risk = 100 * (0.35*drift_component + 0.25*uncertainty_component + 0.40*performance_component)
    health_rows.append({
        "window":name, "mean_PSI":max_psi,
        "uncertainty_component":uncertainty_component,
        "performance_drop_component":performance_component,
        "experimental_risk_score":risk,
        "health_band": "Investigate" if risk >= 50 else ("Watch" if risk >= 25 else "Stable")
    })
health_df=pd.DataFrame(health_rows)
display(health_df.round(3))

In [ ]:
plt.figure(figsize=(10, 5))
colors = health_df["health_band"].map({"Stable":"#35a66f","Watch":"#e5a33b","Investigate":"#d9534f"})
plt.bar(health_df["window"], health_df["experimental_risk_score"], color=colors)
plt.axhline(25, color="#e5a33b", ls="--", label="Watch threshold")
plt.axhline(50, color="#d9534f", ls="--", label="Investigate threshold")
plt.ylim(0, 100)
plt.ylabel("Experimental risk score (0–100)")
plt.title("SIGNAL-SHIFT model health monitor")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Build an automated incident-style report

In [ ]:
def build_incident_report(window_name):
    drift = drift_results[window_name].sort_values("PSI", ascending=False)
    monitor = monitor_df.loc[monitor_df.window == window_name].iloc[0]
    health = health_df.loc[health_df.window == window_name].iloc[0]
    high = drift[drift.PSI >= .25]["feature"].tolist()
    moderate = drift[(drift.PSI >= .1) & (drift.PSI < .25)]["feature"].tolist()

    lines = [
        f"SIGNAL-SHIFT MONITORING REPORT — {window_name}",
        "="*62,
        f"Experimental health band: {health.health_band}",
        f"Experimental risk score: {health.experimental_risk_score:.1f}/100",
        "",
        "MODEL BEHAVIOUR",
        f"F1: {monitor.f1:.3f} | ROC-AUC: {monitor.roc_auc:.3f}",
        f"Alert rate: {monitor.alert_rate:.1%}",
        f"Uncertain prediction rate: {monitor.uncertain_rate:.1%}",
        "",
        "DRIFT SIGNALS",
        f"High PSI features (>=0.25 heuristic): {', '.join(high) if high else 'None'}",
        f"Moderate PSI features (0.10–0.25 heuristic): {', '.join(moderate) if moderate else 'None'}",
        "",
        "RECOMMENDED INVESTIGATION",
        "1. Verify whether the incoming traffic source or collection pipeline changed.",
        "2. Inspect the top drifting features and compare their raw distributions.",
        "3. Check label quality and class prevalence when delayed labels arrive.",
        "4. Evaluate the model on a representative, time-separated labelled sample.",
        "5. Do not retrain automatically from drift alone; investigate root cause first.",
        "",
        "LIMITATION",
        "This report is generated from synthetic data and experimental thresholds. "
        "It is not a production alert or a security incident determination."
    ]
    return "\\n".join(lines)

selected_window = st = "Window 4"
print(build_incident_report(selected_window))

## 12. Export monitoring artifacts

In [ ]:
# Export compact tables for portfolio/research documentation.
monitor_df.to_csv("signal_shift_window_metrics.csv", index=False)
health_df.to_csv("signal_shift_health_scores.csv", index=False)
psi_matrix.to_csv("signal_shift_psi_matrix.csv")

with open("signal_shift_incident_report.txt", "w", encoding="utf-8") as f:
    f.write(build_incident_report("Window 4"))

print("Exported:")
print("- signal_shift_window_metrics.csv")
print("- signal_shift_health_scores.csv")
print("- signal_shift_psi_matrix.csv")
print("- signal_shift_incident_report.txt")

## 13. Critical interpretation: what did the experiment establish?

In [ ]:
# Compact evidence summary derived from the computed experiment.
summary = pd.DataFrame({
    "window": monitor_df["window"],
    "F1": monitor_df["f1"],
    "ROC_AUC": monitor_df["roc_auc"],
    "mean_PSI": [drift_results[n]["PSI"].mean() for n in monitor_df["window"]],
    "max_PSI": [drift_results[n]["PSI"].max() for n in monitor_df["window"]],
    "uncertain_rate": monitor_df["uncertain_rate"],
    "health_band": health_df["health_band"]
})
display(summary.round(3))

### Interpretation guide

When reviewing the results, answer these questions in your own words:

1. Which simulated regime produced the largest feature drift?
2. Did the largest PSI correspond to the largest F1 decline?
3. Which features moved most in the timing-shift window?
4. Did the model become more uncertain in the mixed-shift window?
5. Which monitoring signal would you trust least on its own, and why?
6. What additional evidence would you need before retraining?

### Key scientific cautions
- **Data drift is not concept drift.** The input distribution can change while the relationship between inputs and labels remains stable.
- **Concept drift** means the relationship between inputs and target changes; detecting it generally requires reliable labels or specialized methods.
- **KS p-values** can become tiny for large samples even when the practical effect is small.
- **PSI bins and thresholds** are design choices. Validate them against your actual data and operational needs.
- **Confidence is not correctness.** A model can be confidently wrong.
- **Synthetic results are not evidence of real-world intrusion-detection performance.**

## 14. Advanced extensions
To take SIGNAL-SHIFT beyond this notebook:
- Add rolling windows and streaming ingestion.
- Add ADWIN or Page-Hinkley change detection.
- Add calibration monitoring and Brier score by time window.
- Add delayed-label tracking and concept-drift tests.
- Add SHAP drift: compare feature-attribution patterns over time.
- Add alert suppression, incident history, and audit trails.
- Build a Streamlit monitoring dashboard.
- Evaluate on a real, licensed network intrusion dataset with a documented temporal split.

## Final takeaway

**A model is not reliable just because it performed well yesterday. Reliability is the ability to recognize when yesterday's assumptions may no longer hold.**

That is the central idea behind SIGNAL-SHIFT.